# 00 — Environment tests and GPU pilot

**Purpose:** Check free GPU access, local safety tests, packages, tokenizer alignment, and optional 2-step LoRA/full timing pilots.

**Platform:** Kaggle or Colab (Python + GPU for model stages). Never use held-out test data for model selection.

In [ ]:
from pathlib import Path
import os, sys
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/Urdu_Qwen_Research_Notebooks'), Path('/content/Urdu_Qwen_Research_Notebooks')]
ROOT = next((p for p in CANDIDATES if (p / 'project_utils.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Unzip the entire project folder into /kaggle/working/ or /content/, then run this notebook. See README.md.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from project_utils import *
folders()
print('Project folder:', ROOT)

## A. Environment and safety checks

Run this notebook **first**. The first three cells work without downloading Qwen. If packages are missing, run the installation cell in Kaggle/Colab (restart the kernel once after installation). This notebook does not generate paper results.

In [ ]:
import sys, platform, importlib.util
print('Python:', sys.version.split()[0], 'OS:', platform.platform())
for pkg in ['torch','transformers','peft','accelerate','datasets','sklearn','numpy','pandas']:
    spec = importlib.util.find_spec(pkg)
    print(f'{pkg:15}', 'installed' if spec else 'MISSING')
try:
    import torch
    print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0)); print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))
except ImportError:
    print('Install PyTorch in Kaggle/Colab using the platform-default build.')

### Install only if packages are missing

Kaggle: turn **Internet ON** and **GPU ON**. Colab: Runtime → Change runtime type → T4 GPU. Pip should not replace the platform's CUDA-aware PyTorch.

In [ ]:
# Run this cell ONCE in Kaggle/Colab if libraries are missing. Then restart the kernel.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')])
    print('Restart kernel once before running the model cells.')

In [ ]:
# Offline unit tests: no model download or GPU required.
import subprocess
command = [sys.executable, '-m', 'pytest', '-q', str(ROOT/'tests')]
res = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
print(res.stdout[-4000:], res.stderr[-1000:]); assert res.returncode == 0, 'Unit tests failed'

## B. Optional tokenizer and model smoke test

Set `RUN_MODEL_SMOKE=True` only once packages are installed and internet is available. This loads the real 0.5B model and checks that exactly the answer tokens receive loss.

In [ ]:
RUN_MODEL_SMOKE = False
if RUN_MODEL_SMOKE:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg()['model_id'])
    labels = DEFAULT_LABELS
    example = encode_supervised(tokenizer, 'پاکستان نے کرکٹ میچ جیت لیا', 'sports', labels, 'ur', max_length=256)
    assert len(example['input_ids']) == len(example['labels'])
    assert any(x != -100 for x in example['labels'])
    print('Prompt tokens ignored:', sum(x == -100 for x in example['labels']))
    print('Supervised tokens:',sum(x != -100 for x in example['labels']))
    model, tok = load_model(cfg()['model_id'])
    import torch
    with torch.no_grad():
        batch = supervised_collator(tok)([example]); batch = {k:v.to(next(model.parameters()).device) for k,v in batch.items()}
        loss = model(**batch).loss
    print('Real model loss:',float(loss)); assert torch.isfinite(loss).item()
    del model; torch.cuda.empty_cache() if torch.cuda.is_available() else None

## C. Optional two-step GPU training pilot

Run `01_preprocess_data` first. **This is not a scientific result**: it measures real training speed and checks whether full fine-tuning fits. Test LoRA first, then full. Stop and reduce `config.json → max_length` to 192 or 128 if full training runs out of GPU memory. Use two training steps, not an entire epoch. Both pilots use the **same Adafactor optimizer**, so the optimizer is not a hidden difference.

In [ ]:
RUN_GPU_PILOT = False  # Set True after 01_preprocess_data and GPU enabled
if RUN_GPU_PILOT:
    import torch
    assert torch.cuda.is_available(), 'GPU required for timing pilot'
    for method in ['lora','full']:
        try:
            r = train_run(method, 'sib200', size=5, seed=42, rank=8, max_steps=2, pilot=True,
                          run_name=f'pilot_{method}_{int(__import__("time").time())}')
            print(method, '2 steps:', r['wall_seconds'], 'seconds;', 'VRAM GiB:', r['peak_gpu_allocated_gib'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            print(f'{method}: OOM. Reduce sequence length / keep batch=1, or try another available GPU.')

## D. Runtime estimation after pilot

A rough estimate from a two-step run can be misleading because it includes model loading, initialization, and warmup. Prefer a 10–20-step pilot for better numbers. No result should be reported as a measured full-training time until that training actually completes.

In [ ]:
pilots = [r for r in read_jsonl(ROOT/'outputs/metrics/train_runs.jsonl') if r.get('pilot')]
for r in pilots:
    print(r['run_id'], 'steps=',r['global_steps'], 'total wall min=',round(r['wall_seconds']/60,1),
          'avg sec/step incl. setup=',round(r['wall_seconds']/max(1,r['global_steps']),2))